# BD-KDD Dataset — Exploratory Data Analysis

## Objective

Perform a systematic, reproducible exploratory data analysis of the raw BD-KDD clinical dataset before any DODA-based feature selection or predictive modeling.

### Analysis goals

- Verify the structure and integrity of the raw dataset
- Audit missing values and hidden missing-value representations
- Check duplicate records and identifier behavior
- Characterize numerical and binary/ordinal variables
- Inspect the target distribution
- Examine distributions, ranges, and potential outliers
- Compare feature behavior across target classes
- Investigate correlations and potential redundancy
- Screen for possible data leakage
- Document data-quality decisions before cleaning
- Create and validate a separate processed dataset

**Important:** The raw dataset is never modified by this notebook. All cleaning is performed on a copy and the result is saved separately.


## 1. Imports

Only libraries required for EDA, visualization, and data-quality auditing are imported here.


In [ ]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

# Plotting settings
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


## 2. Project Paths

The confirmed dataset location is used directly.

The raw CSV remains untouched. The cleaned dataset will be written to the existing `processed` directory.


In [ ]:
# ============================================
# 2. PROJECT PATHS
# ============================================

# Confirmed project data location
DATA_ROOT = Path(r"C:\Users\johnm\msc_research\notebooks")

RAW_DIR = DATA_ROOT / "data" / "raw"
PROCESSED_DIR = DATA_ROOT / "data" / "processed"

DATA_PATH = RAW_DIR / "BD-KDD Dataset.csv"
PROCESSED_PATH = PROCESSED_DIR / "BD-KDD_cleaned.csv"

print("Raw data path       :", DATA_PATH)
print("Processed data path :", PROCESSED_PATH)
print("Raw file exists     :", DATA_PATH.exists())
print("Processed directory :", PROCESSED_DIR.exists())


## 3. Load Raw Dataset

The CSV is loaded exactly as provided. No values are changed at this stage.


In [ ]:
# ============================================
# 3. LOAD RAW DATASET
# ============================================

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")


## 4. Initial Dataset Inspection

First inspect the raw observations and schema before making any preprocessing decisions.


In [ ]:
# ============================================
# 4.1 FIRST 5 ROWS
# ============================================

display(df.head())


In [ ]:
# ============================================
# 4.2 LAST 5 ROWS
# ============================================

display(df.tail())


In [ ]:
# ============================================
# 4.3 DATASET DIMENSIONS
# ============================================

print(f"Number of rows    : {df.shape[0]:,}")
print(f"Number of columns : {df.shape[1]:,}")


In [ ]:
# ============================================
# 4.4 COLUMN NAMES
# ============================================

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")


In [ ]:
# ============================================
# 4.5 DATA TYPES
# ============================================

display(df.dtypes.to_frame(name="Data Type"))


In [ ]:
# ============================================
# 4.6 DATASET INFORMATION
# ============================================

df.info()


## 5. Missing-Value Audit

Missingness is checked in several ways:

1. Pandas-recognized missing values
2. Hidden textual missing-value tokens
3. Row-level missingness

No imputation or deletion is performed during this section.


In [ ]:
# ============================================
# 5.1 COLUMN-LEVEL MISSING VALUES
# ============================================

missing_count = df.isna().sum()
missing_percent = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percent
}).sort_values("Missing Count", ascending=False)

display(missing_summary)


In [ ]:
# ============================================
# 5.2 TOTAL MISSING VALUES
# ============================================

print("Total missing values:", int(df.isna().sum().sum()))


In [ ]:
# ============================================
# 5.3 HIDDEN MISSING-VALUE TOKENS
# ============================================

missing_tokens = [
    "?", "NA", "N/A", "na", "n/a",
    "Unknown", "unknown", "-", ""
]

hidden_missing = {}

for token in missing_tokens:
    count = (df.astype(str) == token).sum().sum()
    if count > 0:
        hidden_missing[token] = int(count)

if hidden_missing:
    for token, count in hidden_missing.items():
        print(f"{token!r}: {count} occurrence(s)")
else:
    print("No predefined hidden missing-value tokens detected.")


In [ ]:
# ============================================
# 5.4 ROW-LEVEL MISSINGNESS
# ============================================

missing_by_row = df.isna().sum(axis=1)

print("Rows containing missing values :", int((missing_by_row > 0).sum()))
print("Maximum missing values in row :", int(missing_by_row.max()))


## 6. Duplicate and Identifier Audit

Exact duplicate rows and identifier behavior are assessed separately.

`Sl. No.` is treated as a potential identifier, not as a clinical predictor, until its behavior is verified.


In [ ]:
# ============================================
# 6.1 EXACT DUPLICATE ROWS
# ============================================

duplicate_count = int(df.duplicated().sum())

print("Exact duplicate rows :", duplicate_count)
print(f"Duplicate percentage : {duplicate_count / len(df) * 100:.2f}%")


In [ ]:
# ============================================
# 6.2 IDENTIFIER AUDIT
# ============================================

id_col = "Sl. No."

print("Unique IDs :", df[id_col].nunique())
print("Total rows :", len(df))
print("Duplicate IDs :", int(df[id_col].duplicated().sum()))

print("\nFirst 10 IDs:")
display(df[id_col].head(10).to_frame())


In [ ]:
# ============================================
# 6.3 ID SEQUENCE CHECK
# ============================================

expected_ids = np.arange(1, len(df) + 1)

print("IDs are exactly 1..N:",
      bool(np.array_equal(df[id_col].to_numpy(), expected_ids)))


## 7. Variable Inventory

The dataset contains numerical measurements as well as binary/ordinal encoded clinical variables.

The table below documents the analytical role of each variable without changing the raw column names.


In [ ]:
# ============================================
# 7. VARIABLE INVENTORY
# ============================================

variable_inventory = pd.DataFrame([
    ["Sl. No.", "Identifier", "Administrative", "Exclude from modeling"],
    ["Age", "Numerical", "Demographic", "Predictor"],
    ["Bp", "Numerical", "Vital sign", "Predictor"],
    ["Sg", "Ordinal", "Urinalysis", "Predictor"],
    ["Al", "Ordinal", "Urinalysis", "Predictor"],
    ["Su", "Ordinal", "Urinalysis", "Predictor"],
    ["Rbc", "Binary encoded", "Urinalysis", "Predictor"],
    ["Pc", "Binary encoded", "Urinalysis", "Predictor"],
    ["Pcc", "Binary encoded", "Urinalysis", "Predictor"],
    ["Ba", "Binary encoded", "Urinalysis", "Predictor"],
    ["Bgr", "Numerical", "Blood laboratory", "Predictor"],
    ["Bu", "Numerical", "Renal laboratory", "Predictor"],
    ["Sc", "Numerical", "Renal laboratory", "Predictor"],
    ["Sod", "Numerical", "Electrolyte", "Predictor"],
    ["Pot", "Numerical", "Electrolyte", "Predictor"],
    ["Hemo", "Numerical", "Hematology", "Predictor"],
    ["Pcv", "Numerical", "Hematology", "Predictor"],
    ["Wbcc", "Numerical", "Hematology", "Predictor"],
    ["Rbcc", "Numerical", "Hematology", "Predictor"],
    ["Htn", "Binary encoded", "Comorbidity", "Predictor"],
    ["Dm", "Binary encoded", "Comorbidity", "Predictor"],
    ["Cad", "Binary encoded", "Comorbidity", "Predictor"],
    ["Appet", "Binary encoded", "Clinical symptom", "Predictor"],
    ["Pe", "Binary encoded", "Clinical symptom", "Predictor"],
    ["Ane", "Binary encoded", "Clinical symptom", "Predictor"],
    ["Class", "Binary target", "Outcome", "Target"],
], columns=["Feature", "Variable Type", "Clinical Group", "Modeling Role"])

display(variable_inventory)


## 8. Unique-Value and Encoding Audit

This section verifies how variables are actually encoded in the CSV.

Binary variables should contain only two distinct values, while ordinal variables such as `Sg`, `Al`, and `Su` should be inspected for their observed levels.


In [ ]:
# ============================================
# 8.1 UNIQUE VALUE COUNTS
# ============================================

unique_summary = pd.DataFrame({
    "Unique Values": df.nunique(dropna=False),
    "Data Type": df.dtypes.astype(str)
})

display(unique_summary)


In [ ]:
# ============================================
# 8.2 VALUES OF LOW-CARDINALITY VARIABLES
# ============================================

low_cardinality_cols = [
    "Sg", "Al", "Su",
    "Rbc", "Pc", "Pcc", "Ba",
    "Htn", "Dm", "Cad",
    "Appet", "Pe", "Ane",
    "Class"
]

for col in low_cardinality_cols:
    print(f"{col}: {sorted(df[col].unique().tolist())}")


In [ ]:
# ============================================
# 8.3 BINARY VARIABLE VALIDATION
# ============================================

binary_cols = [
    "Rbc", "Pc", "Pcc", "Ba",
    "Htn", "Dm", "Cad",
    "Appet", "Pe", "Ane"
]

binary_check = pd.DataFrame({
    "Unique Values": [df[col].nunique() for col in binary_cols],
    "Observed Values": [sorted(df[col].unique().tolist()) for col in binary_cols]
}, index=binary_cols)

display(binary_check)


## 9. Target Analysis

`Class` is inspected as the outcome variable.

The labels are kept as encoded in the dataset. Their clinical meaning will be reported according to the dataset documentation rather than inferred from the numeric values alone.


In [ ]:
# ============================================
# 9.1 TARGET COUNTS
# ============================================

target_counts = df["Class"].value_counts(dropna=False).sort_index()

display(target_counts.to_frame(name="Count"))


In [ ]:
# ============================================
# 9.2 TARGET DISTRIBUTION
# ============================================

target_distribution = pd.DataFrame({
    "Count": df["Class"].value_counts(dropna=False).sort_index(),
    "Percentage": (
        df["Class"]
        .value_counts(normalize=True, dropna=False)
        .sort_index() * 100
    )
})

display(target_distribution)


In [ ]:
# ============================================
# 9.3 TARGET PLOT
# ============================================

plt.figure(figsize=(6, 4))

ax = sns.countplot(data=df, x="Class")

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.title("Target Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()


## 10. Numerical Feature Analysis

Descriptive statistics are calculated for numerical predictors.

This is an inspection stage only. No scaling, normalization, or outlier removal is performed here.


In [ ]:
# ============================================
# 10.1 NUMERICAL FEATURES
# ============================================

numerical_cols = [
    "Age", "Bp", "Bgr", "Bu", "Sc",
    "Sod", "Pot", "Hemo", "Pcv",
    "Wbcc", "Rbcc"
]

display(df[numerical_cols].describe().T)


In [ ]:
# ============================================
# 10.2 RANGE SUMMARY
# ============================================

range_summary = pd.DataFrame({
    "Minimum": df[numerical_cols].min(),
    "Maximum": df[numerical_cols].max(),
    "Range": df[numerical_cols].max() - df[numerical_cols].min(),
    "Unique Values": df[numerical_cols].nunique()
})

display(range_summary)


In [ ]:
# ============================================
# 10.3 ZERO AND NEGATIVE VALUE CHECK
# ============================================

zero_counts = (df[numerical_cols] == 0).sum()
negative_counts = (df[numerical_cols] < 0).sum()

range_flags = pd.DataFrame({
    "Zero Count": zero_counts,
    "Negative Count": negative_counts
})

display(range_flags)


## 11. Numerical Distributions

Histograms are used to inspect the shape and spread of continuous/numerical variables.

These plots are exploratory and are not used by themselves to declare values invalid.


In [ ]:
# ============================================
# 11.1 NUMERICAL DISTRIBUTIONS
# ============================================

n_cols = 3
n_rows = int(np.ceil(len(numerical_cols) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 4 * n_rows)
)

axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, numerical_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

for ax in axes[len(numerical_cols):]:
    ax.remove()

plt.tight_layout()
plt.show()


## 12. Categorical / Binary Feature Analysis

Frequency tables are generated for the binary and ordinal variables.

This allows us to verify class representation and detect unexpected encodings.


In [ ]:
# ============================================
# 12.1 FREQUENCY TABLES
# ============================================

categorical_cols = [
    "Sg", "Al", "Su",
    "Rbc", "Pc", "Pcc", "Ba",
    "Htn", "Dm", "Cad",
    "Appet", "Pe", "Ane"
]

for col in categorical_cols:
    print(f"\n{'=' * 60}")
    print(f"{col}")
    print(f"{'=' * 60}")

    counts = df[col].value_counts(dropna=False).sort_index()
    percentages = (
        df[col].value_counts(normalize=True, dropna=False)
        .sort_index() * 100
    )

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages
    })

    display(summary)


## 13. Clinical Plausibility Screening

This section flags observations that deserve inspection based on simple numerical rules.

**Important:** A statistical extreme is not automatically a clinical error. These checks identify records for review; they do not automatically delete or correct observations.

Clinical interpretation should be based on the dataset's documentation and appropriate clinical references.


In [ ]:
# ============================================
# 13.1 SIMPLE PLAUSIBILITY FLAGS
# ============================================

plausibility_rules = {
    "Age": lambda s: s < 0,
    "Bp": lambda s: s <= 0,
    "Sg": lambda s: (s <= 0),
    "Bgr": lambda s: s < 0,
    "Bu": lambda s: s < 0,
    "Sc": lambda s: s < 0,
    "Sod": lambda s: s <= 0,
    "Pot": lambda s: s <= 0,
    "Hemo": lambda s: s <= 0,
    "Pcv": lambda s: s <= 0,
    "Wbcc": lambda s: s <= 0,
    "Rbcc": lambda s: s <= 0,
}

plausibility_summary = []

for col, rule in plausibility_rules.items():
    mask = rule(df[col])
    plausibility_summary.append({
        "Feature": col,
        "Flagged Records": int(mask.sum()),
        "Flag Percentage": float(mask.mean() * 100)
    })

plausibility_summary = pd.DataFrame(plausibility_summary)

display(plausibility_summary)


In [ ]:
# ============================================
# 13.2 OBSERVED LEVELS FOR ORDINAL VARIABLES
# ============================================

ordinal_cols = ["Sg", "Al", "Su"]

ordinal_levels = pd.DataFrame({
    col: pd.Series(sorted(df[col].unique()))
    for col in ordinal_cols
})

display(ordinal_levels)


## 14. Outlier Screening

The IQR rule is used only as a statistical screening method.

Outliers will **not** be removed automatically because clinically severe patients can legitimately produce extreme measurements.


In [ ]:
# ============================================
# 14.1 IQR OUTLIER SUMMARY
# ============================================

outlier_rows = []

for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (df[col] < lower) | (df[col] > upper)

    outlier_rows.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outlier Count": int(mask.sum()),
        "Outlier Percentage": float(mask.mean() * 100)
    })

outlier_summary = pd.DataFrame(outlier_rows)

display(outlier_summary)


In [ ]:
# ============================================
# 14.2 BOXPLOTS
# ============================================

n_cols = 3
n_rows = int(np.ceil(len(numerical_cols) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 4 * n_rows)
)

axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, numerical_cols):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(f"Boxplot of {col}")

for ax in axes[len(numerical_cols):]:
    ax.remove()

plt.tight_layout()
plt.show()


## 15. Class-Wise Feature Analysis

Feature distributions are compared between the two target classes.

This is descriptive analysis only. These comparisons are not used as a substitute for formal statistical testing or feature selection.


In [ ]:
# ============================================
# 15.1 CLASS-WISE NUMERICAL SUMMARY
# ============================================

classwise_summary = (
    df.groupby("Class")[numerical_cols]
    .agg(["mean", "median", "std"])
)

display(classwise_summary)


In [ ]:
# ============================================
# 15.2 CLASS-WISE NUMERICAL BOXPLOTS
# ============================================

n_cols = 3
n_rows = int(np.ceil(len(numerical_cols) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 4 * n_rows)
)

axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, numerical_cols):
    sns.boxplot(data=df, x="Class", y=col, ax=ax)
    ax.set_title(f"{col} by Class")

for ax in axes[len(numerical_cols):]:
    ax.remove()

plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 15.3 BINARY FEATURE BY TARGET
# ============================================

binary_cols = [
    "Rbc", "Pc", "Pcc", "Ba",
    "Htn", "Dm", "Cad",
    "Appet", "Pe", "Ane"
]

for col in binary_cols:
    table = pd.crosstab(
        df[col],
        df["Class"],
        normalize="columns"
    ) * 100

    print(f"\n{col} (% within target class)")
    display(table)


## 16. Correlation and Feature Redundancy

Pearson correlation is used as an exploratory measure among numerical variables.

Correlation does not establish causation and does not determine whether a feature should be retained. It is used here to identify potentially redundant measurements that may affect feature-selection behavior.


In [ ]:
# ============================================
# 16.1 NUMERICAL CORRELATION MATRIX
# ============================================

correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 9))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)

plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 16.2 HIGH-CORRELATION PAIRS
# ============================================

corr_pairs = (
    correlation_matrix
    .where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

high_corr_pairs = corr_pairs[abs(corr_pairs) >= 0.70]

print("Pairs with |correlation| >= 0.70:")
display(high_corr_pairs.to_frame("Correlation"))


## 17. Leakage and Identifier Checks

Potential leakage is considered before feature selection.

The main questions are:

- Is `Sl. No.` merely an identifier?
- Does any predictor exactly reproduce the target?
- Are there duplicate columns?
- Are there suspiciously deterministic relationships?
- Is any feature likely to represent a post-diagnosis label rather than a predictor?

This section is a screening step, not proof that no leakage exists.


In [ ]:
# ============================================
# 17.1 TARGET DUPLICATION CHECK
# ============================================

feature_cols_all = [col for col in df.columns if col != "Class"]

exact_target_matches = []

for col in feature_cols_all:
    if df[col].equals(df["Class"]):
        exact_target_matches.append(col)

print("Features exactly identical to target:")
print(exact_target_matches if exact_target_matches else "None")


In [ ]:
# ============================================
# 17.2 DUPLICATE COLUMN CHECK
# ============================================

duplicate_columns = []

for i in range(len(df.columns)):
    for j in range(i + 1, len(df.columns)):
        if df.iloc[:, i].equals(df.iloc[:, j]):
            duplicate_columns.append(
                (df.columns[i], df.columns[j])
            )

print("Duplicate column pairs:")
print(duplicate_columns if duplicate_columns else "None")


In [ ]:
# ============================================
# 17.3 TARGET ASSOCIATION SCREENING
# ============================================

target_correlations = (
    df[numerical_cols + ["Class"]]
    .corr(numeric_only=True)["Class"]
    .drop("Class")
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

display(target_correlations.to_frame("Correlation with Class"))


## 18. Data Quality Summary

This section records the observed properties of the raw dataset before cleaning.

The values below are generated from the actual CSV, rather than copied from the dataset description.


In [ ]:
# ============================================
# 18. DATA QUALITY SUMMARY
# ============================================

quality_summary = {
    "Rows": len(df),
    "Columns": df.shape[1],
    "Missing Values": int(df.isna().sum().sum()),
    "Exact Duplicate Rows": int(df.duplicated().sum()),
    "Duplicate IDs": int(df[id_col].duplicated().sum()),
    "Unique Target Values": int(df["Class"].nunique()),
    "Target Class 0 Count": int((df["Class"] == 0).sum()),
    "Target Class 1 Count": int((df["Class"] == 1).sum()),
}

quality_summary_df = pd.DataFrame(
    quality_summary.items(),
    columns=["Check", "Result"]
)

display(quality_summary_df)


## 19. Cleaning Decisions

Based on the audit above, cleaning decisions are made explicitly.

### Planned decisions

1. Preserve the raw dataset unchanged.
2. Remove `Sl. No.` from the modeling dataset because it is an administrative identifier rather than a clinical predictor.
3. Do not impute values if the raw audit confirms no missing values.
4. Do not remove statistical outliers automatically.
5. Do not apply scaling or normalization in the EDA cleaning step.
6. Preserve the original clinical variable encodings.
7. Preserve the original target encoding.
8. Any future transformation, encoding, scaling, or feature engineering will be performed inside the modeling pipeline to avoid data leakage.

These decisions can be revised if later auditing reveals a genuine data-quality problem.


In [ ]:
# ============================================
# 19. CREATE CLEANED DATASET
# ============================================

df_clean = df.copy()

# Remove administrative identifier from modeling dataset
df_clean = df_clean.drop(columns=["Sl. No."])

print("Original shape :", df.shape)
print("Cleaned shape  :", df_clean.shape)

print("\nColumns removed:")
print(set(df.columns) - set(df_clean.columns))


## 20. Validate Processed Dataset

Before saving, verify that the cleaned dataset has not accidentally introduced missing values, duplicates, or unexpected target changes.


In [ ]:
# ============================================
# 20.1 PROCESSED DATA VALIDATION
# ============================================

print("Missing values       :", int(df_clean.isna().sum().sum()))
print("Duplicate rows       :", int(df_clean.duplicated().sum()))
print("Target values        :", sorted(df_clean["Class"].unique().tolist()))
print("Number of rows       :", len(df_clean))
print("Number of predictors :", df_clean.shape[1] - 1)


In [ ]:
# ============================================
# 20.2 TARGET PRESERVATION CHECK
# ============================================

original_target = df["Class"].value_counts().sort_index()
cleaned_target = df_clean["Class"].value_counts().sort_index()

target_check = pd.DataFrame({
    "Original": original_target,
    "Processed": cleaned_target
})

display(target_check)

print(
    "Target distribution preserved:",
    bool(original_target.equals(cleaned_target))
)


## 21. Save Processed Dataset

The cleaned dataset is saved separately from the raw CSV.

**Raw data remains untouched.**


In [ ]:
# ============================================
# 21. SAVE PROCESSED DATASET
# ============================================

df_clean.to_csv(PROCESSED_PATH, index=False)

print("Processed dataset saved successfully.")
print(PROCESSED_PATH)
print("File exists:", PROCESSED_PATH.exists())


In [ ]:
# ============================================
# 21.1 FINAL RELOAD CHECK
# ============================================

df_processed_check = pd.read_csv(PROCESSED_PATH)

print("Reloaded processed dataset successfully.")
print(f"Shape: {df_processed_check.shape[0]:,} rows × {df_processed_check.shape[1]:,} columns")

display(df_processed_check.head())


# EDA Complete

The raw BD-KDD dataset has been inspected without modification.

The processed dataset has been created only after the data-quality audit and validation steps.

### Next research stage

The processed dataset can now be used for:

1. Train/test splitting
2. Cross-validation design
3. Baseline preprocessing pipeline
4. Statistical feature selection
5. DODA clinical weighting
6. Downstream model evaluation
7. Feature stability analysis
8. Explainability analysis

**Feature selection and model preprocessing should be performed within the training folds during cross-validation to prevent information leakage.**
